# Stateful AI Agent Architecture & Persistent Memory with LangGraph

**Tech Stack:** Python 3.14 | LangGraph | LangChain Core | Google Gemini (`ChatGoogleGenerativeAI`) | SQLite (`SqliteSaver`) | Pydantic  

---

## Architecture Overview
A stateful AI agent architecture engineered with **LangGraph**, **Google Gemini**, and **SQLite** to manage multi-turn conversations efficiently.

Unlike stateless chains that resend full conversational history with quadratic token growth, this architecture integrates:
- **Typed State & Reducers (`add_messages`):** Deterministic state updates and message appending via `MessagesState`.
- **Dynamic Context Condensation (`RemoveMessage`):** Autonomous LLM-driven dialogue summarization and token eviction when history exceeds thresholds.
- **Conditional Agent Pipeline:** Specialized nodes for Gemini reasoning and autonomous context distillation.
- **Persistent Memory (`SqliteSaver`):** ACID-compliant thread checkpoints (`thread_id`) and time-travel state auditing (`get_state_history`).
- **Interactive Gradio UI:** Live multi-turn chat interface (`gr.ChatInterface`) with real-time state and summary visibility.


---
## 1. Environment Configuration & Library Imports

In this section, I configure the environment and import core primitives from **LangGraph** (state graph construction, state schemas, message reducers, and SQLite checkpointers) alongside **LangChain Core** (message schemas) and **LangChain Google GenAI** (`ChatGoogleGenerativeAI`).


In [1]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

import os
import sqlite3
from typing import Literal
from typing_extensions import TypedDict
from collections.abc import Sequence

# LangChain message primitives
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    RemoveMessage
)

# LangGraph core primitives
from langgraph.graph import (
    START,
    END,
    StateGraph,
    MessagesState,
    add_messages
)

# Memory and Checkpointing
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

# Language Model (Google Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI


---
## 2. Comprehensive State Schema Definition

Here, I designed a strongly-typed `AgentState` schema inheriting from LangGraph's `MessagesState`.

By extending `MessagesState`, the `messages` list automatically leverages the `add_messages` reducer to safely append incoming dialogue and process deletions via `RemoveMessage`. I introduced an additional `summary: str` field to persist high-level conversational memory across conversation cycles without bloating the prompt window.


In [2]:
class AgentState(MessagesState):
    """
    Comprehensive Agent State:
    - messages: Sequence[BaseMessage] managed by add_messages reducer
    - summary: str containing LLM-condensed context of earlier conversation
    """
    summary: str


---
## 3. LLM Model Initialization

I initialized Google Gemini using `ChatGoogleGenerativeAI` (configurable via `GEMINI_MODEL`, defaulting to `gemini-2.5-flash`) with a balanced temperature setting (`0.7`) to deliver fluent, context-aware responses and reliable dialogue summarization.


In [3]:
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

chat = ChatGoogleGenerativeAI(
    model=gemini_model,
    google_api_key=gemini_api_key,
    temperature=0.7
)


---
## 4. Agent Node Implementations

I implemented 2 specialized, decoupled nodes to handle reasoning and memory optimization:
1. `chatbot`: Injects any existing cumulative summary as a `SystemMessage` context anchor, appends active message history, and queries Google Gemini.
2. `summarize_messages`: An autonomous context optimizer that invokes Gemini to merge active dialogue into the cumulative `summary` and emits `RemoveMessage` tokens to prune historical messages from active memory.


In [4]:
def chatbot(state: AgentState) -> AgentState:
    """Node 1: Generates assistant response, incorporating cumulative summary if present."""
    print("\n======> [NODE: chatbot]")
    
    current_summary = state.get("summary", "")
    messages_to_send = []
    
    if current_summary:
        system_instruction = f'''
Here is a concise summary of the conversation so far:
{current_summary}

Keep this context in mind when answering the following user message.
'''
        messages_to_send.append(SystemMessage(system_instruction))
    
    # Filter valid non-empty messages
    valid_messages = []
    for msg in state.get("messages", []):
        text = getattr(msg, "text", "") or getattr(msg, "content", "")
        if isinstance(text, str) and text.strip():
            valid_messages.append(msg)
        elif isinstance(text, list) and text:
            valid_messages.append(msg)
    
    # Gemini requires the final turn to be a user message (no assistant prefilling)
    if not valid_messages or not isinstance(valid_messages[-1], HumanMessage):
        valid_messages.append(HumanMessage("Please continue."))
        
    messages_to_send.extend(valid_messages)
    response = chat.invoke(messages_to_send)
    
    assistant_text = getattr(response, "text", "") or (response.content if isinstance(response.content, str) else str(response.content))
    print(f"Assistant: {assistant_text}")
    return AgentState(messages=[response])


In [5]:
def summarize_messages(state: AgentState) -> AgentState:
    """Node 2: Compresses conversation history into cumulative summary and prunes messages."""
    print("\n======> [NODE: summarize_messages]")
    
    # Format active conversation history
    conversation_text = ""
    for msg in state["messages"]:
        msg_text = getattr(msg, "text", "") or (msg.content if isinstance(msg.content, str) else str(msg.content))
        conversation_text += f"{msg.type.upper()}: {msg_text}\n\n"
    
    summary_prompt = f'''
Update the ongoing conversation summary by merging the new lines of dialogue below with the previous summary.
Preserve all key user details, names, facts, and answered topics from the previous summary rather than dropping or replacing them.
Respond ONLY with the updated concise summary text.

Previous Summary:
{state.get("summary", "")}

New Dialogue:
{conversation_text}
'''
    updated_summary = chat.invoke([HumanMessage(summary_prompt)])
    summary_text = getattr(updated_summary, "text", "") or (updated_summary.content if isinstance(updated_summary.content, str) else str(updated_summary.content))
    print(f"[SUMMARY UPDATED]: {summary_text[:120]}...")
    
    # Evict messages that have been summarized using RemoveMessage
    remove_tokens = [RemoveMessage(id=msg.id) for msg in state["messages"] if hasattr(msg, "id") and msg.id]
    
    return AgentState(
        messages=remove_tokens,
        summary=summary_text
    )


---
## 5. Dynamic Routing Function & Conditional Edges

To enable autonomous context management, I engineered a conditional routing function `routing_function`:
- **Context Threshold Check:** If active dialogue exceeds **4 messages**, it routes to `summarize_messages` to compress history and evict old messages before ending the turn.
- **Direct Completion:** If active dialogue is within limits (`<= 4` messages), it routes directly to `__end__`.


In [6]:
def routing_function(state: AgentState) -> Literal["summarize_messages", "__end__"]:
    """Inspects active message count to decide whether to summarize before finishing the turn."""
    if len(state.get("messages", [])) > 4:
        return "summarize_messages"
    return "__end__"


---
## 6. Graph Construction & Persistent Checkpointing

In this step, I assembled the `StateGraph` topology, wired sequential and conditional routing edges, and compiled the graph with an ACID-compliant SQLite checkpointer (`SqliteSaver`).

This persistent checkpointer ensures that conversations survive process interruptions, scale across multi-tenant sessions, and maintain state integrity across unique `thread_id` identifiers.


In [7]:
# Initialize StateGraph
graph = StateGraph(AgentState)

# Add Nodes
graph.add_node("chatbot", chatbot)
graph.add_node("summarize_messages", summarize_messages)

# Connect Edges
graph.add_edge(START, "chatbot")
graph.add_conditional_edges(
    source="chatbot",
    path=routing_function
)
graph.add_edge("summarize_messages", END)

# Connect Persistent Checkpointer (SQLite)
db_connection = sqlite3.connect("langgraph_memory.db", check_same_thread=False)
checkpointer = SqliteSaver(db_connection)

# Compile Graph
agent_app = graph.compile(checkpointer=checkpointer)
print("StateGraph compiled successfully with SQLite checkpointing!")


StateGraph compiled successfully with SQLite checkpointing!


---
## 7. Graph Topology Visualization

To verify graph integrity and edge transitions, I rendered the compiled execution topology in ASCII representation.


In [8]:
try:
    print(agent_app.get_graph().draw_ascii())
except ImportError:
    print("Graph topology compiled successfully. (Restart the kernel to render the ASCII diagram: Kernel -> Restart)")


                   +-----------+          
                   | __start__ |          
                   +-----------+          
                         *                
                         *                
                         *                
                    +---------+           
                    | chatbot |           
                    +---------+.          
                 ...            ...       
               ..                  ..     
             ..                      ..   
+--------------------+                 .. 
| summarize_messages |               ..   
+--------------------+             ..     
                 ***            ...       
                    **        ..          
                      **    ..            
                    +---------+           
                    | __end__ |           
                    +---------+           


---
## 8. Interactive Gradio Chat UI & Thread Execution

Here, I wired a lightweight `gradio.ChatInterface` directly to `agent_app` under an isolated session configuration (`thread_id="session_001"`).

Each user turn streams node updates (`chatbot` → conditional `summarize_messages` → `__end__`) into SQLite and displays the live active message count and cumulative `summary` right below the reply.


In [9]:
import gradio as gr

config_thread_1 = {"configurable": {"thread_id": "session_001"}}

def gradio_chat(message: str, history: list) -> str:
    """Handle one user message from the Gradio chat window using persistent LangGraph state."""
    ai_reply = ""
    summarized = False

    # Stream node updates so we capture chatbot's reply even when summarize_messages prunes messages
    for update in agent_app.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config_thread_1,
        stream_mode="updates",
    ):
        if "chatbot" in update:
            last_msg = update["chatbot"]["messages"][-1]
            ai_reply = getattr(last_msg, "text", "") or (
                last_msg.content if isinstance(last_msg.content, str) else str(last_msg.content)
            )
        if "summarize_messages" in update:
            summarized = True

    latest_state = agent_app.get_state(config_thread_1).values
    active_count = len(latest_state.get("messages", []))
    summary = latest_state.get("summary", "")

    footer = (
        f"\n\n---\n"
        f"*Thread: `{config_thread_1['configurable']['thread_id']}` | "
        f"Active messages in state: `{active_count}` | "
        f"Condensed this turn: `{'Yes' if summarized else 'No'}`*"
    )
    if summary:
        footer += f"\n> **Persistent Summary:** {summary}"
    return ai_reply + footer

gradio_ui = gr.ChatInterface(
    fn=gradio_chat,
    title="LangGraph Stateful Agent & Persistent Memory",
    description="Multi-turn Gemini agent with dynamic RemoveMessage context pruning (>4 messages) and SQLite persistence.",
    analytics_enabled=False,
)

print("Gradio chatbot interface is ready.")
gradio_ui.launch()


C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio chatbot interface is ready.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---
## 9. State History Inspection & Time-Travel Auditing

One of the most powerful features I incorporated is state observability. By leveraging `get_state_history()`, we can inspect the full chronological trajectory of checkpoints, examine intermediate state snapshots, and perform time-travel debugging across the agent's entire execution lifecycle.


In [11]:
# Retrieve all historical snapshots for this thread
history = list(agent_app.get_state_history(config_thread_1))
print(f"Total historical checkpoints recorded for session_001: {len(history)}\n")

for i, snapshot in enumerate(history[::-1]):
    print(f"Checkpoint {i+1}: Node '{snapshot.next}' | Values: {list(snapshot.values.keys())}")
    if snapshot.values.get("summary"):
        print(f"   -> Summary: {snapshot.values['summary'][:80]}...")


Total historical checkpoints recorded for session_001: 10

Checkpoint 1: Node '('__start__',)' | Values: ['messages']
Checkpoint 2: Node '('chatbot',)' | Values: ['messages']
Checkpoint 3: Node '()' | Values: ['messages']
Checkpoint 4: Node '('__start__',)' | Values: ['messages']
Checkpoint 5: Node '('chatbot',)' | Values: ['messages']
Checkpoint 6: Node '()' | Values: ['messages']
Checkpoint 7: Node '('__start__',)' | Values: ['messages']
Checkpoint 8: Node '('chatbot',)' | Values: ['messages']
Checkpoint 9: Node '('summarize_messages',)' | Values: ['messages']
Checkpoint 10: Node '()' | Values: ['messages', 'summary']
   -> Summary: User Name: Burraq
Project: Building a stateful LangGraph agent using SQLite.
Dis...
